In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

import sys
import os
import pickle
from tqdm import tqdm
# Get the absolute path to the folder containing `utils`
utils_path = os.path.abspath('../')
if utils_path not in sys.path:
    sys.path.append(utils_path)
    
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # second gpu
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
    
from utils import *

# az.style.use("arviz-docgrid")
plt.rcParams['figure.dpi'] = 140

experiment_orientations = [159, 123, 87, 51, 15]
subjects = ["01", "02", "03", "04", "05", "06", "07" ,"09", "10", "11", "12"]
median_key = {15:0, 51:1, 87:2, 123:3, 159:4}
std_key = {15:0, 51:1, 87:2, 123:3, 159:4}

c_table = pd.read_csv('../data_caches/ctiraltable.csv')
med = np.load('../data_caches/med.npy')
std = np.load('../data_caches/std.npy')
ctimetable = np.load('../data_caches/ctimetable.npy')
r_table = pd.read_csv('../data_caches/rtrialtable.csv', index_col=0)
(x, y, d, r, e, cd, ce) = np.load('../data_caches/rtimetable.npy', allow_pickle=True)

In [2]:
import jax
jax.config.update('jax_platform_name', 'gpu')

import jax.numpy as jnp
import jax.random as jr
from jax import lax
from jax import vmap
import optax

from jax.extend import backend
print(backend.get_backend().platform)

gpu


In [3]:
xaxis = np.arange(-250, 750, 1) * (1000/120)
start_idx, end_idx = np.searchsorted(xaxis, -500), np.searchsorted(xaxis, 1500)
(start_idx, end_idx)

# our frmes of intrest are only
er = e[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

# ds
ds = d[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

er[np.isnan(er)] = 90
ds[np.isnan(ds)] = 0

# min max scale err
er = er / 180

emissions = jnp.array(np.stack([ds, er], axis=-1))

er.shape, np.isnan(er).any(), ds.shape, np.isnan(ds).any(), emissions.shape

((34560, 239), False, (34560, 239), False, (34560, 239, 2))

In [4]:
num_states, emission_dim = 1, 2

In [5]:
# model = GaussianModel(emission_dim)
# parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior")
# fit_params, ll = model.fit_em(parameters, properties, emissions[:2880], num_iters=1000, verbose=True)
# plt.plot(ll)

In [6]:
crossval_results = {}
sd = emissions.reshape(12, 4 * 6 * 120, 239, 2)

for idx in tqdm(range(0, 12)):
    crossval_results[idx] = {}
    
    subset_idx = np.random.choice(np.arange(0, 2880), 2880, replace=False)
    ss_em = sd[idx][subset_idx]
    
    model = GaussianModel(output_dim = emission_dim)
    ll_mean, ll = cross_validate_dist(model=model, emissions=ss_em, key=jr.PRNGKey(0), num_iters=1000, init = "default", num_folds=100)
    crossval_results[idx][num_states] = ll

100%|██████████| 12/12 [18:22<00:00, 91.88s/it]


In [8]:
with open('./caches/one_state_crossval_result.pkl', 'wb') as f:
    pickle.dump(crossval_results, f)